# ExaHyPE 2 Tutorials - Part 2: The Shallow Water Equations

In Part 1 everything was handed to you. This time **you** implement the physics.
The Python configuration below is ready to run, but the C++ file
**`ADERDGSolver.cpp`** contains only stubs marked with `TODO`. Your job is to
fill them in, rebuild, and watch a dam break.

This tutorial introduces two new ingredients on top of Part 1:

* a **non-conservative product** (NCP) for the bathymetry term, and
* **static mesh refinement** driven by a refinement criterion.

## The Shallow Water Equations

The shallow water equations model a thin layer of fluid. The unknowns are the
water height $h$, the momenta $h u_1, h u_2$ and the bathymetry (sea-bed
height) $b$:

$$
\frac{\partial}{\partial t}
\begin{pmatrix} h \\ h u_1 \\ h u_2 \\ b \end{pmatrix}
+ \nabla \cdot
\begin{pmatrix}
 h u_1 & h u_2 \\
 h u_1^2 + \tfrac{1}{2} g h^2 & h u_1 u_2 \\
 h u_2 u_1 & h u_2^2 + \tfrac{1}{2} g h^2 \\
 0 & 0
\end{pmatrix} = \mathbf{0},
\qquad g = 9.81 .
$$

The wave speeds (eigenvalues in direction $n$) are $u_n \pm \sqrt{g h}$ and
$u_n$, so the largest eigenvalue by magnitude is
$\max\!\big(|u_n + \sqrt{g h}|,\ |u_n - \sqrt{g h}|\big)$ with
$u_n = (hu)_n / h$.

We start without bathymetry (a flat bed) and add it as the NCP later.

## Importing the API

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath("../../api/"))
import exahype2

## Defining the Project

A $2 \times 2$ box, this time with **non-periodic** boundaries (the dam break
runs into walls), named `DamBreak`.

In [ ]:
my_project = exahype2.Project(
    namespace    = ["tutorials", "exahype2", "swe"],
    project_name = "DamBreak",
    executable   = "DamBreak",
    dimensions   = 2,
    size         = [ 2.0,  2.0],
    offset       = [-1.0, -1.0],
    periodic_bc  = [False, False],
)

## Defining the Solver

Again an ADER-DG solver named `ADERDGSolver`, now with four unknowns. Compared
with Part 1 we declare more PDE terms in `set_implementation`, because the
shallow water system needs them:

* `flux` and `max_eigenvalue` as before;
* `ncp` - the non-conservative product $B(Q)\,\nabla Q$ (used for the
  bathymetry term in the extension exercise);
* `refinement_criterion` - lets us refine the mesh over the bathymetry bump.

In [ ]:
aderdg_solver = exahype2.solvers.aderdg.GlobalAdaptiveTimeStep(
    name                 = "ADERDGSolver",
    order                = 3,
    time_step_relaxation = 0.9,
    unknowns             = {"h": 1, "hu": 2, "b": 1},
    auxiliary_variables  = {},
)

aderdg_solver.set_implementation(
    flux                 = exahype2.solvers.PDETerms.User_Defined_Implementation,
    ncp                  = exahype2.solvers.PDETerms.User_Defined_Implementation,
    refinement_criterion = exahype2.solvers.PDETerms.User_Defined_Implementation,
    max_eigenvalue       = exahype2.solvers.PDETerms.User_Defined_Implementation,
)

my_project.add_solver(aderdg_solver)

## Building the Project

Build it once now. It will compile (the stubs are valid C++) but the simulation
will do nothing physical until you implement the TODOs - re-run this cell
after each change to `ADERDGSolver.cpp`.

In [ ]:
my_project.output.makefile.add_cpp_file("ADERDGSolver.cpp")
my_project.generate_and_build()

## Exercise 1 - A Dam Break

Open **`ADERDGSolver.cpp`** and complete the TODOs. The variables are addressed
through the generated `Shortcuts` enum (`Shortcuts::h`, `Shortcuts::hu + 0/1`,
`Shortcuts::b`).

1. **`initialCondition`** - a dam break: water at rest ($hu = 0$), bed
   $b = 0$, and a height step, e.g. $h = 1$ on the left half of the domain
   ($x_0 < 0$) and $h = 2$ on the right. The position is in `cellCentre`.
2. **`boundaryConditions`** - impose a calm outside state: $h = 1$,
   $hu = 0$, $b = 0$ in `Qoutside`.
3. **`maxEigenvalue`** - return
   $\max(|u_n + \sqrt{g h}|, |u_n - \sqrt{g h}|)$ with $g = 9.81$ and
   $u_n = $ `Q[Shortcuts::hu + normal] / Q[Shortcuts::h]`.
4. **`flux`** - the flux above (keep the $\tfrac12 g h^2$ pressure term in
   the flux for now). Leave the NCP zero for Exercise 1.

Rebuild (the cell above) after editing, then run the cell below. A worked
solution is on the documentation page *Part 2 - Solution to Exercise 1*
if you get stuck.

## Running the Simulation

`-lmin` and `-lmax` set the coarsest and finest grid level. For Exercise 1, which has no refinement criterion yet, **keep them equal** for a uniform grid; in Exercise 2 you let them differ so ExaHyPE refines only where the criterion asks. Edit the parameters in the run cell below and re-run - no rebuild needed.

In [ ]:
import os, subprocess

# Runtime parameters - edit these freely, no rebuild needed.
end_time  = 0.5     # -et   : physical end time
snapshots = 20      # -ns   : number of output snapshots (drives the animation)
min_level = 4       # -lmin : coarsest mesh level
max_level = 4       # -lmax : finest mesh level (equal to -lmin -> uniform grid)

# Run with 4 OpenMP threads (the notebook equivalent of: export OMP_NUM_THREADS=4)
os.environ["OMP_NUM_THREADS"] = "4"

# Exercise 1 runs on a uniform grid (min_level == max_level). The dam break is a
# sharp step and this scheme has no limiter yet (that is Part 4), so it needs
# enough resolution: level 4 resolves the front cleanly but takes a few minutes;
# level 3 is faster for a first look, just visibly coarser. Coarser still and
# the time step collapses. In Exercise 2 you add a refinement criterion and run
# with min_level = 2, max_level = 3, so the mesh is fine only where needed.
subprocess.run(
    ["./DamBreak", "-et", str(end_time), "-ns", str(snapshots), "-lmin", str(min_level), "-lmax", str(max_level)],
    check=True,
)

## Visualising the Result

Watch the water height `h` as the step collapses and waves bounce off the walls.
If the picture is flat, the physics TODOs are not in yet.

In [ ]:
import importlib, os, sys
sys.path.insert(0, os.path.abspath("../../"))
from postprocessing import render_solution
importlib.reload(render_solution)
render_solution.show_solution("ADERDGSolver", field="h")

## Exercise 2 - Bathymetry and Static Refinement

With bathymetry the gravity term moves out of the flux and into the
**non-conservative product**:

$$
\nabla\cdot
\begin{pmatrix} h u_1 & h u_2 \\ h u_1^2 & h u_1 u_2 \\ h u_2 u_1 & h u_2^2 \\ 0 & 0\end{pmatrix}
+
\underbrace{\begin{pmatrix} 0 \\ g h\,\partial_x(h+b) \\ g h\,\partial_y(h+b) \\ 0 \end{pmatrix}}_{B(Q)\,\nabla Q}
= \mathbf{0}.
$$

5. **`flux`** - remove the $\tfrac12 g h^2$ term you added in step 4 (it now
   lives in the NCP).
6. **`nonconservativeProduct`** - implement $B(Q)\,\nabla Q$. The gradient
   of the state in the `normal` direction is supplied in `deltaQ`, so the
   momentum component is `g * Q[Shortcuts::h] * (deltaQ[Shortcuts::h] +
   deltaQ[Shortcuts::b])`.
7. **`initialCondition`** - switch to a *radial* setup: $h = 1$
   everywhere and a circular bump in the bed, $b = 0.1$ within distance $0.5$ of
   the origin (use `tarch::la::norm2(cellCentre)`), else $b = 0$.
8. **`refinementCriterion`** - return `::exahype2::RefinementCommand::Refine`
   in a fixed disc over the bump (e.g. `::tarch::la::norm2(cellCentre) < 0.3`),
   otherwise `Keep`. The criterion looks only at position, so the refined patch
   is **static** - a fixed fine region over the bump, never moving.

Rebuild, then **set `min_level = 2` and `max_level = 3` in the run cell
above**. Now that a refinement criterion exists, the two levels may differ:
ExaHyPE keeps a coarse level-2 base mesh and refines to level 3 only inside
the disc - fine resolution over the bump, a cheap coarse far field everywhere
else. Re-run and watch the water drain off the bump into a central dip.

A worked solution is on the documentation page *Part 2 - Solution to Exercise 2* if you get stuck.

When you are happy, continue with **Part 3 - The Euler Equations over an Airfoil**, which uses a
completely different numerical scheme.